# XML 


In [1]:
import os
import time
import requests
import pandas as pd
from tqdm import tqdm

# Caminho completo para o CSV original do BGG
CSV_PATH = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\boardgames_ranks.csv"

# Diretório onde os XMLs serão guardados (mesma pasta do CSV)
XML_DIR = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\xml_raw"
os.makedirs(XML_DIR, exist_ok=True)

def download_game_xml(game_id, overwrite=False, delay=2):
    """
    Faz download do XML de um jogo via BGG API e guarda localmente.
    
    Args:
        game_id (int): ID do jogo na BGG.
        overwrite (bool): Se True, sobrescreve ficheiros já existentes.
        delay (int): Delay entre pedidos para evitar bloqueio da API.
    """
    filepath = os.path.join(XML_DIR, f"{game_id}.xml")

    # Ignorar se já existir
    if not overwrite and os.path.exists(filepath):
        return "exists"

    url = f"https://boardgamegeek.com/xmlapi2/thing?id={game_id}&stats=1"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            with open(filepath, "wb") as f:
                f.write(response.content)
            time.sleep(delay)
            return "ok"
        elif response.status_code == 429:
            # Rate limit da API atingido
            time.sleep(10)
            return "retry"
        else:
            return f"error_{response.status_code}"
    except Exception as e:
        return f"exception_{str(e)}"


# Leitura dos IDs e extração incremental dos XMLs

In [7]:
#!pip install nest_asyncio
!pip install aiohttp nest_asyncio tqdm pandas


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\marco\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import asyncio
import aiohttp
import nest_asyncio
import json
import os
import pandas as pd
import time
from tqdm import tqdm
from collections import Counter

# Aplicar patch para permitir asyncio dentro do Jupyter/VS Code
nest_asyncio.apply()

# Caminhos
PROGRESS_FILE = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\progress_bgg.json"
XML_DIR = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\xml_raw"
CSV_PATH = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\boardgames_ranks.csv"

# Configurações
MAX_CONCURRENT = 10     # número de downloads simultâneos (5–10 é seguro)
RETRY_LIMIT = 3         # nº de tentativas antes de desistir
SAVE_EVERY = 200        # guardar progresso a cada N jogos

os.makedirs(XML_DIR, exist_ok=True)

# -----------------------------
# 1. Carregar dataset e progresso
# -----------------------------
print("A carregar ficheiro base...")
df_ranks = pd.read_csv(CSV_PATH)
ids = df_ranks["id"].dropna().unique().astype(int).tolist()

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
        done_ids = set(json.load(f))
    print(f"Progresso anterior carregado ({len(done_ids):,} jogos já descarregados)")
else:
    done_ids = set()
    print("Nenhum progresso anterior encontrado. Início do zero.")

remaining_ids = [gid for gid in ids if gid not in done_ids]
print(f"Total de jogos por descarregar: {len(remaining_ids):,}")

# -----------------------------
# 2. Função assíncrona de download individual
# -----------------------------
async def fetch_game_xml(session, game_id):
    url = f"https://boardgamegeek.com/xmlapi2/thing?id={game_id}&stats=1"
    filepath = os.path.join(XML_DIR, f"{game_id}.xml")

    # Ignorar se já existir
    if os.path.exists(filepath):
        return "exists"

    for attempt in range(RETRY_LIMIT):
        try:
            async with session.get(url) as response:
                if response.status == 200:
                    content = await response.read()
                    with open(filepath, "wb") as f:
                        f.write(content)
                    return "ok"
                elif response.status == 429:  # Too many requests
                    await asyncio.sleep(10)
                else:
                    return f"error_{response.status}"
        except Exception as e:
            if attempt < RETRY_LIMIT - 1:
                await asyncio.sleep(5)
            else:
                return f"exception_{str(e)}"

# -----------------------------
# 3. Função principal (controlo e progresso)
# -----------------------------
async def download_all(ids):
    connector = aiohttp.TCPConnector(limit_per_host=MAX_CONCURRENT)
    timeout = aiohttp.ClientTimeout(total=None)
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        results = {}
        tasks = []
        sem = asyncio.Semaphore(MAX_CONCURRENT)
        pbar = tqdm(total=len(ids), desc="A descarregar XMLs (async)")

        async def bounded_fetch(gid):
            async with sem:
                status = await fetch_game_xml(session, gid)
                results[gid] = status
                pbar.update(1)

        for i, gid in enumerate(ids):
            tasks.append(asyncio.create_task(bounded_fetch(gid)))

            # Guardar progresso a cada N jogos
            if (i + 1) % SAVE_EVERY == 0:
                await asyncio.gather(*tasks)
                done_ids.update([k for k, v in results.items() if v in ["ok", "exists"]])
                with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
                    json.dump(list(done_ids), f)
                results.clear()
                tasks.clear()
                print(f"Progresso guardado: {len(done_ids):,} jogos")
                await asyncio.sleep(1)

        # Processar o resto
        await asyncio.gather(*tasks)
        pbar.close()
        done_ids.update([k for k, v in results.items() if v in ["ok", "exists"]])
        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump(list(done_ids), f)
        return results

# -----------------------------
# 4. Executar o processo completo
# -----------------------------
loop = asyncio.get_event_loop()
start_time = time.time()
results = loop.run_until_complete(download_all(remaining_ids))
elapsed = time.time() - start_time

print("\nExtração concluída.")
print(f"Total processados: {len(done_ids):,}")
print(f"Duração total: {elapsed/3600:.2f} horas")
print(f"XMLs guardados em: {XML_DIR}")

# Resumo geral
Counter(results.values())


A carregar ficheiro base...
Progresso anterior carregado (167,041 jogos já descarregados)
Total de jogos por descarregar: 0


A descarregar XMLs (async): 0it [00:00, ?it/s]


Extração concluída.
Total processados: 167,041
Duração total: 0.00 horas
XMLs guardados em: C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\xml_raw


Counter()